---
## 📊 PARTIE 12 — Indice Trimestriel Désaisonnalisé (Option C — STL Trend)

### Principe
On réutilise la **composante tendance STL** déjà calculée en Partie 11.  
La tendance STL est par définition **désaisonnalisée** — elle ne contient plus le cycle annuel.  
On la rééchantillonne à fréquence **trimestrielle** pour obtenir ~36 points par zone.

### Avantages vs indice annuel
| Critère | Annuel | Trimestriel STL |
|---------|--------|----------------|
| Points par zone | 9 | ~36 |
| Saisonnalité | Non contrôlée | ✓ Éliminée par STL |
| Stabilité | ✓ Bonne | ✓ Bonne (lissage STL) |
| Tests statistiques | Limités (n=9) | Plus puissants (n=36) |
| Comparabilité inter-zones | ✓ | ✓ |

### Workflow
1. Extraire tendance STL → rééchantillonner Q1/Q2/Q3/Q4
2. Convertir en fraction urbaine % via seuil -14 dB (calibration sur l'annuel)
3. Visualiser : séries trimestrielles + heatmap + carte choroplèthe animée
4. Tests : Mann-Kendall trimestriel, accélération/décélération de croissance
5. Indice ville trimestriel pondéré superficie

### 📥 Cellule A — Extraction Tendance STL Trimestrielle

In [ ]:
# ── CELL 12A — Tendance STL → Rééchantillonnage Trimestriel ────────────────
# Pas de nouveau appel GEE — on réutilise stl_results de la Partie 11

import pandas as pd
import numpy as np
from scipy import stats

print('Extraction tendance STL trimestrielle...')
print('(Aucun appel GEE — réutilisation des résultats STL Partie 11)')
print()

YEARS_MONTHLY  = list(range(2016, 2025))
full_idx = pd.date_range(f'{YEARS_MONTHLY[0]}-01-01',
                          f'{YEARS_MONTHLY[-1]}-12-01', freq='MS')

records_q = []

for zone in ZONE_ORDER:
    res = stl_results[zone]

    # Série tendance STL sur index mensuel complet
    ts_trend = pd.Series(res.trend, index=full_idx, name='trend_vv')

    # Rééchantillonnage trimestriel — moyenne de la tendance sur le trimestre
    ts_q = ts_trend.resample('QS').mean()   # QS = début de trimestre

    for date, vv_trend in ts_q.items():
        if np.isnan(vv_trend):
            continue
        quarter = (date.month - 1) // 3 + 1
        records_q.append({
            'zone'     : zone,
            'date'     : date,
            'year'     : date.year,
            'quarter'  : quarter,
            'label'    : f'{date.year}-Q{quarter}',
            'vv_trend' : round(float(vv_trend), 4),
        })

df_q = pd.DataFrame(records_q)

# ── Calibration VV trend → Fraction Urbaine % ───────────────────────────────
# On cale sur l'indice annuel déjà calculé (uf_2024, uf_annual_by_zone)
# Méthode : régression linéaire VV_trend_annuel ~ UF_annuel par zone
# Pour chaque zone, on interpole entre le min et max de la tendance

print('Calibration VV trend → Fraction Urbaine %')
print('Méthode : mapping linéaire zone par zone calé sur UF annuel')
print()

# Reconstruire UF annuel moyen par zone depuis df_annual (Partie 6)
# On utilise le mapping VV → UF via les points ancres connus :
#   VV = -14 dB → UF = 0%  (seuil de classification)
#   VV = -8 dB  → UF ≈ 97% (Arafat, zone la plus urbanisée)
# Transformation linéaire : UF = clip((VV - (-14)) / (-8 - (-14)) * 97, 0, 100)

VV_ZERO  = -14.0   # seuil non-urbain
VV_MAX   = -8.0    # VV d'une zone 100% béton dense
UF_MAX   = 97.0    # fraction max observée (Arafat)

def vv_to_uf(vv):
    """Conversion VV trend (dB) → fraction urbaine estimée (%)"""
    uf = (vv - VV_ZERO) / (VV_MAX - VV_ZERO) * UF_MAX
    return float(np.clip(uf, 0, 100))

df_q['uf_trend'] = df_q['vv_trend'].apply(vv_to_uf)

# Vérification de cohérence avec l'annuel
print('Vérification cohérence UF trimestrielle vs UF annuelle :')
print(f'{"Zone":20s} {"UF_annuel_2024":>15s} {"UF_trim_2024":>14s} {"Écart":>8s}')
print('-'*62)
for zone in ZONE_ORDER:
    sub_q   = df_q[(df_q['zone']==zone) & (df_q['year']==2024)]
    uf_q_24 = sub_q['uf_trend'].mean() if len(sub_q)>0 else np.nan
    uf_a_24 = uf_2024.get(zone, np.nan)
    ecart   = uf_q_24 - uf_a_24 if not np.isnan(uf_q_24) else np.nan
    print(f'{zone:20s} {uf_a_24:>15.1f} {uf_q_24:>14.1f} {ecart:>+8.1f}')

df_q.to_csv('/outputs/quarterly_uf_stl.csv', index=False)
print()
print(f'✓ {len(df_q)} enregistrements trimestriels')
print(f'  Zones : {df_q["zone"].nunique()}')
print(f'  Période : {df_q["label"].iloc[0]} → {df_q["label"].iloc[-1]}')
print('✓ Saved: /outputs/quarterly_uf_stl.csv')

### 📈 Cellule B — Séries Trimestrielles par Moughataa

In [ ]:
# ── CELL 12B — Visualisation Séries Trimestrielles ──────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(22, 16))
fig.suptitle('Indice Urbain Trimestriel Désaisonnalisé — Nouakchott 2016-2024\n'
             'Basé sur la composante Tendance STL  |  Saisonnalité éliminée',
             fontsize=14, fontweight='bold')

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub   = df_q[df_q['zone']==zone].sort_values('date')
    color = ZONE_COLORS[zone]

    # Série UF trimestrielle
    ax.plot(sub['date'], sub['uf_trend'], color=color, lw=2.5,
            marker='o', ms=4, zorder=3, label='UF trimestrielle')

    # Intervalle de confiance ±1 std glissant (fenêtre 4 trimestres)
    rolling_std = sub['uf_trend'].rolling(4, center=True, min_periods=2).std()
    ax.fill_between(sub['date'],
                    sub['uf_trend'] - rolling_std,
                    sub['uf_trend'] + rolling_std,
                    color=color, alpha=0.12, zorder=1, label='±1σ glissant')

    # Points annuels de référence (indice annuel original)
    for year in YEARS_MONTHLY:
        uf_a = uf_annual_by_zone.get(zone, {}).get(year, None)
        if uf_a:
            ax.scatter(pd.Timestamp(f'{year}-07-01'), uf_a,
                      marker='D', s=40, color='black', zorder=5, alpha=0.6)

    # Régression linéaire tendance
    x_num  = np.arange(len(sub))
    slope, intercept, r, p, _ = stats.linregress(x_num, sub['uf_trend'])
    trend_line = intercept + slope * x_num
    ax.plot(sub['date'], trend_line, color=color, lw=1.5,
            ls='--', alpha=0.7, zorder=2)

    # Annotation slope
    slope_yr = slope * 4  # trimestres → annuel
    sig      = '✓' if p < 0.05 else 'ns'
    ax.text(0.03, 0.06,
            f'+{slope_yr:.2f} pp/an  (p={p:.3f} {sig})',
            transform=ax.transAxes, fontsize=8, color=color,
            bbox=dict(facecolor='white', alpha=0.75, boxstyle='round', linewidth=0))

    ax.set_title(zone, fontsize=10, fontweight='bold', color=color)
    ax.set_ylabel('Fraction Urbaine (%)', fontsize=8)
    ax.set_ylim(bottom=max(0, sub['uf_trend'].min()-5))
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2, ls=':')
    ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=7, rotation=45)

# Légende diamants
axes[0,0].scatter([], [], marker='D', s=40, color='black', alpha=0.6,
                  label='Indice annuel (référence)')
axes[0,0].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('/outputs/quarterly_01_series.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_01_series.png')

### 🗓️ Cellule C — Heatmap Trimestrielle + Détection Accélération

In [ ]:
# ── CELL 12C — Heatmap Trimestrielle et Accélération de Croissance ─────────
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('Indice Trimestriel — Heatmap et Accélération de Croissance\n'
             'Nouakchott 2016-2024  |  Désaisonnalisé STL',
             fontsize=13, fontweight='bold')

# ── Heatmap Zone × Trimestre ─────────────────────────────────────
ax = axes[0]
pivot_q = df_q.pivot_table(index='zone', columns='label',
                            values='uf_trend', aggfunc='mean')

# Réordonner zones par UF 2024 décroissant
zone_order_uf = sorted(ZONE_ORDER, key=lambda z: uf_2024.get(z, 0), reverse=True)
pivot_q = pivot_q.reindex(zone_order_uf)

cmap_hm = LinearSegmentedColormap.from_list('uf_hm',
    ['#f0f4f8','#c9d6e3','#7fb3d3','#2c7bb6','#1a3a5c'])

im = ax.imshow(pivot_q.values, aspect='auto', cmap=cmap_hm,
               vmin=0, vmax=100, interpolation='nearest')

# Annotations valeurs (tous les 2 trimestres pour lisibilité)
for i in range(len(pivot_q.index)):
    for j in range(len(pivot_q.columns)):
        if j % 2 == 0:
            val = pivot_q.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                        fontsize=5.5,
                        color='white' if val > 50 else 'black')

# Labels axes
ax.set_yticks(range(len(zone_order_uf)))
ax.set_yticklabels(zone_order_uf, fontsize=8)
# Labels X : seulement les Q1 (début d'année)
q_labels  = list(pivot_q.columns)
q1_idx    = [i for i, l in enumerate(q_labels) if 'Q1' in l]
q1_labels = [l.replace('-Q1','') for l in q_labels if 'Q1' in l]
ax.set_xticks(q1_idx)
ax.set_xticklabels(q1_labels, fontsize=8, rotation=45)
ax.set_title('Fraction Urbaine % par Zone et Trimestre', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='UF (%)')

# ── Accélération : slope par sous-période ────────────────────────
ax2 = axes[1]

# Diviser en 3 sous-périodes : 2016-2018 / 2019-2021 / 2022-2024
periods = [
    ('2016-2018', 2016, 2018, '#457b9d'),
    ('2019-2021', 2019, 2021, '#2a9d8f'),
    ('2022-2024', 2022, 2024, '#e63946'),
]

slopes_by_period = {z: {} for z in ZONE_ORDER}
for label_p, y1, y2, color_p in periods:
    for zone in ZONE_ORDER:
        sub = df_q[(df_q['zone']==zone) &
                   (df_q['year']>=y1) & (df_q['year']<=y2)].sort_values('date')
        if len(sub) < 4:
            continue
        x = np.arange(len(sub))
        slope, _, _, p, _ = stats.linregress(x, sub['uf_trend'])
        slopes_by_period[zone][label_p] = slope * 4  # → pp/an

x_pos    = np.arange(len(ZONE_ORDER))
width    = 0.25
for i, (label_p, y1, y2, color_p) in enumerate(periods):
    vals = [slopes_by_period[z].get(label_p, 0) for z in ZONE_ORDER]
    bars = ax2.bar(x_pos + i*width, vals, width, label=label_p,
                   color=color_p, alpha=0.85, edgecolor='white')

ax2.axhline(0, color='black', lw=1, ls='-')
ax2.set_xticks(x_pos + width)
ax2.set_xticklabels(ZONE_ORDER, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Vitesse de croissance (pp/an)', fontsize=10)
ax2.set_title('Accélération / Décélération par Période\n'
              '(pente de régression sur sous-période)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.25, ls=':')

# Annotation : zone la plus accélérée sur 2022-2024
fastest = max(ZONE_ORDER,
              key=lambda z: slopes_by_period[z].get('2022-2024', -999))
ax2.text(0.98, 0.97,
         f'Croissance la plus rapide\n2022-2024 : {fastest}',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9,
         bbox=dict(facecolor='#e63946', alpha=0.15, boxstyle='round'))

plt.tight_layout()
plt.savefig('/outputs/quarterly_02_heatmap_acceleration.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_02_heatmap_acceleration.png')

print()
print('Vitesses de croissance (pp/an) par sous-période :')
print(f'{"Zone":20s} {"2016-18":>10s} {"2019-21":>10s} {"2022-24":>10s} {"Accélération":>14s}')
print('-'*58)
for zone in ZONE_ORDER:
    s1 = slopes_by_period[zone].get('2016-2018', np.nan)
    s2 = slopes_by_period[zone].get('2019-2021', np.nan)
    s3 = slopes_by_period[zone].get('2022-2024', np.nan)
    acc = s3 - s1 if not np.isnan(s1) and not np.isnan(s3) else np.nan
    flag = '↑ accélère' if acc > 0.3 else ('↓ ralentit' if acc < -0.3 else '→ stable')
    print(f'{zone:20s} {s1:>+10.2f} {s2:>+10.2f} {s3:>+10.2f} {flag}')

### 📐 Cellule D — Test Mann-Kendall Trimestriel + Breakpoints

In [ ]:
# ── CELL 12D — Mann-Kendall + Détection de Rupture ──────────────────────────
# Mann-Kendall : test non-paramétrique de tendance monotone
# Plus robuste que OLS pour les séries courtes non-normales

try:
    import pymannkendall as mk
    HAS_MK = True
except ImportError:
    print('pymannkendall non installé — installation...')
    import subprocess
    subprocess.run(['pip', 'install', 'pymannkendall', '--quiet',
                    '--break-system-packages'], check=True)
    import pymannkendall as mk
    HAS_MK = True

print('Test de Mann-Kendall sur les séries trimestrielles désaisonnalisées')
print('H0 : pas de tendance monotone')
print('='*72)
print(f'{"Zone":20s} {"stat Tau":>10s} {"p-value":>10s} {"Tendance":>12s} {"slope":>10s} {"sens":>8s}')
print('-'*72)

mk_results = {}
for zone in ZONE_ORDER:
    sub  = df_q[df_q['zone']==zone].sort_values('date')['uf_trend'].dropna()
    res  = mk.original_test(sub)
    slope_q = res.slope * 4  # trimestre → annuel
    mk_results[zone] = {
        'tau': res.Tau, 'p': res.p, 'trend': res.trend,
        'slope': slope_q, 's': res.s
    }
    sig = ('***' if res.p<0.001 else '**' if res.p<0.01
           else '*' if res.p<0.05 else 'ns')
    print(f'{zone:20s} {res.Tau:>+10.3f} {res.p:>10.4f} '
          f'{res.trend:>12s} {slope_q:>+10.3f} {sig:>8s}')

print()
print('Toutes les zones avec tendance significative :')
sig_zones = [z for z in ZONE_ORDER if mk_results[z]['p'] < 0.05]
print(f'  {len(sig_zones)}/9 zones significatives')
for z in sig_zones:
    print(f'  ✓ {z}: +{mk_results[z]["slope"]:.2f} pp/an (p={mk_results[z]["p"]:.4f})')

# ── Figure Mann-Kendall ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Test de Mann-Kendall — Tendance Trimestrielle Désaisonnalisée\n'
             'Nouakchott 2016-2024  |  Toutes zones', fontsize=13, fontweight='bold')

# Tau values
ax = axes[0]
taus  = [mk_results[z]['tau']  for z in ZONE_ORDER]
pvals = [mk_results[z]['p']    for z in ZONE_ORDER]
cols  = ['#2a9d8f' if p<0.05 else '#adb5bd' for p in pvals]
bars  = ax.barh(ZONE_ORDER, taus, color=cols, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Tau de Kendall  [+1 = tendance croissante parfaite]')
ax.set_title('Tau de Kendall par Moughataa', fontweight='bold')
for bar, p in zip(bars, pvals):
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    if sig:
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                sig, va='center', fontsize=10, color='#2a9d8f', fontweight='bold')
ax.grid(axis='x', alpha=0.3, ls=':')

# Slopes pp/an
ax2 = axes[1]
slopes = [mk_results[z]['slope'] for z in ZONE_ORDER]
cols2  = [ZONE_COLORS[z] for z in ZONE_ORDER]
ax2.barh(ZONE_ORDER, slopes, color=cols2, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='black', lw=1)
ax2.set_xlabel('Vitesse de croissance urbaine (pp/an)')
ax2.set_title('Pente de Theil-Sen (pp/an)', fontweight='bold')
ax2.grid(axis='x', alpha=0.3, ls=':')

plt.tight_layout()
plt.savefig('/outputs/quarterly_03_mann_kendall.png', dpi=150, bbox_inches='tight')
plt.close()
print()
print('✓ Saved: quarterly_03_mann_kendall.png')

### 🌍 Cellule E — Indice Ville Trimestriel + Synthèse Finale

In [ ]:
# ── CELL 12E — Indice Ville Trimestriel + Figure Synthèse ───────────────────

# ── Indice ville pondéré superficie ──────────────────────────────
city_q_records = []
for label_q in df_q['label'].unique():
    sub = df_q[df_q['label']==label_q]
    total_area = 0; weighted_uf = 0
    for zone in ZONE_ORDER:
        row = sub[sub['zone']==zone]
        if len(row) == 0 or np.isnan(row['uf_trend'].values[0]):
            continue
        w = areas[zone]
        weighted_uf += row['uf_trend'].values[0] * w
        total_area  += w
    if total_area > 0:
        date_q = df_q[df_q['label']==label_q]['date'].iloc[0]
        city_q_records.append({
            'label'  : label_q,
            'date'   : date_q,
            'year'   : date_q.year,
            'quarter': (date_q.month-1)//3+1,
            'uf_city': round(weighted_uf/total_area, 3)
        })

df_city_q = pd.DataFrame(city_q_records).sort_values('date')

# Mann-Kendall ville
res_city = mk.original_test(df_city_q['uf_city'].dropna())
slope_city_q = res_city.slope * 4

print(f'Indice ville trimestriel :')
print(f'  Mann-Kendall Tau = {res_city.Tau:+.3f}  p = {res_city.p:.4f}  '
      f'tendance = {res_city.trend}')
print(f'  Pente Theil-Sen  = {slope_city_q:+.3f} pp/an')

# ── Figure synthèse finale 2 panneaux ────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 14),
                          gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle('Indice Urbain Trimestriel Désaisonnalisé — Synthèse Finale\n'
             'Nouakchott 2016-2024  |  Basé sur Tendance STL Sentinel-1 SAR',
             fontsize=14, fontweight='bold')

# ── Panneau 1 : toutes les zones ─────────────────────────────────
ax = axes[0]
for zone in ZONE_ORDER:
    sub   = df_q[df_q['zone']==zone].sort_values('date')
    color = ZONE_COLORS[zone]
    ax.plot(sub['date'], sub['uf_trend'], color=color, lw=1.8,
            alpha=0.85, label=zone)
    # Dernier point annoté
    last = sub.iloc[-1]
    ax.annotate(f"{zone.split()[0]}
{last['uf_trend']:.0f}%",
                (last['date'], last['uf_trend']),
                xytext=(8, 0), textcoords='offset points',
                fontsize=6.5, color=color, va='center')

ax.set_ylabel('Fraction Urbaine Estimée (%)', fontsize=11)
ax.set_title('Trajectoires Trimestrielles par Moughataa', fontweight='bold')
ax.legend(fontsize=8, loc='upper left', ncol=3)
ax.grid(alpha=0.2, ls=':')
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax.tick_params(axis='x', labelsize=9)

# Lignes Q1 verticales (début d'année)
for year in YEARS_MONTHLY:
    ax.axvline(pd.Timestamp(f'{year}-01-01'), color='grey',
               lw=0.5, ls=':', alpha=0.4)

# ── Panneau 2 : indice ville ──────────────────────────────────────
ax2 = axes[1]
ax2.fill_between(df_city_q['date'], df_city_q['uf_city'],
                 alpha=0.15, color='#1d3557')
ax2.plot(df_city_q['date'], df_city_q['uf_city'],
         color='#1d3557', lw=3, marker='o', ms=4, label='Indice ville')

# Tendance linéaire
x_city = np.arange(len(df_city_q))
s_c, i_c, _, _, _ = stats.linregress(x_city, df_city_q['uf_city'])
ax2.plot(df_city_q['date'], i_c + s_c*x_city,
         color='#e63946', lw=2, ls='--', label=f'Tendance ({s_c*4:+.2f} pp/an)')

# Annotations annuelles
for year in YEARS_MONTHLY:
    sub_y = df_city_q[df_city_q['year']==year]
    if len(sub_y) > 0:
        mean_y = sub_y['uf_city'].mean()
        ax2.annotate(f'{mean_y:.1f}%',
                     (pd.Timestamp(f'{year}-07-01'), mean_y),
                     xytext=(0, 8), textcoords='offset points',
                     fontsize=7.5, ha='center', color='#1d3557')

ax2.set_ylabel('Fraction Urbaine Ville (%)', fontsize=11)
ax2.set_title(f'Indice Ville Global — Mann-Kendall : Tau={res_city.Tau:+.3f}  '
              f'p={res_city.p:.4f}  ({res_city.trend})', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2, ls=':')
ax2.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax2.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('/outputs/quarterly_04_final_synthesis.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_04_final_synthesis.png')

print()
print('=== RÉCAPITULATIF PARTIE 12 ===')
print(f'  Indice trimestriel : {len(df_city_q)} points ville  |  '
      f'{len(df_q)} points zone')
print(f'  Tendance ville     : {slope_city_q:+.3f} pp/an  '
      f'(p={res_city.p:.4f}, {res_city.trend})')
print()
for f in ['quarterly_01_series.png',
          'quarterly_02_heatmap_acceleration.png',
          'quarterly_03_mann_kendall.png',
          'quarterly_04_final_synthesis.png',
          'quarterly_uf_stl.csv']:
    print(f'  ✓ /outputs/{f}')